# SegEarth-R1 mask-path fine-tuning with D4 augmentation (Kaggle, GPU T4 x2)
ft_d4 vs ft_ctrl (no augmentation), full validation split + 8-way TTA.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import sys, torch; print(sys.version.split()[0], '| torch', torch.__version__, '| CUDA GPUs:', torch.cuda.device_count())" 

In [ ]:
import os
os.environ['USE_FLAX'] = '0'; os.environ['USE_TF'] = '0'    # PyTorch only (Kaggle's JAX needs NumPy 2; we keep Kaggle's NumPy)
!pip -q install transformers==4.36.2 tokenizers==0.15.2 timm==0.6.13 fvcore addict einops einops-exts==0.0.4 accelerate==0.21.0 peft==0.4.0 tabulate 2>&1 | tail -2
!python -c "import numpy, transformers, timm, torch; print('numpy', numpy.__version__, '| transformers', transformers.__version__, '| timm', timm.__version__, '| torch', torch.__version__)" 

In [ ]:
import glob, os, shutil, zipfile
# Code, weights, data and the 5.6 GB feature cache live in /kaggle/tmp (not saved). Only results/ is saved:
# it is a link into /kaggle/working, so even a crashed run leaves a small, downloadable output.
W = '/kaggle/tmp/Minor'; os.makedirs(W, exist_ok=True)
zips = glob.glob('/kaggle/input/**/bac_code.zip', recursive=True)
if zips:                                    # dataset kept the zip
    zipfile.ZipFile(zips[0]).extractall(W)
else:                                       # Kaggle auto-extracted the zip: copy the extracted tree
    marker = glob.glob('/kaggle/input/**/bac/common.py', recursive=True)
    assert marker, 'Add the bac-code dataset via Add Input'
    shutil.copytree(os.path.dirname(os.path.dirname(marker[0])), W, dirs_exist_ok=True)
os.makedirs('/kaggle/working/results/logs', exist_ok=True)
if not os.path.islink(f'{W}/results'):
    shutil.rmtree(f'{W}/results', ignore_errors=True); os.symlink('/kaggle/working/results', f'{W}/results')
os.chdir(W)
!ls; df -h /kaggle/working /kaggle/tmp | cat

In [ ]:
# downloads run in a subprocess so the pinned libraries are used everywhere
!python -c "from huggingface_hub import snapshot_download; snapshot_download('earth-insights/SegEarth-R1-EarthReason', local_dir='checkpoints/SegEarth-R1-EarthReason')"
!python -c "from huggingface_hub import hf_hub_download; import zipfile; [zipfile.ZipFile(hf_hub_download('earth-insights/EarthReason', f'{s}.zip', repo_type='dataset', local_dir='/kaggle/tmp')).extractall('data/rs_reason_seg/RSReasonSeg') for s in ('val', 'train')]"
!ls data/rs_reason_seg/RSReasonSeg/train/images | wc -l; ls data/rs_reason_seg/RSReasonSeg/val/images | wc -l; du -sh checkpoints

In [ ]:
# fail fast: if the environment is broken, stop here instead of skipping through hours of runs
import subprocess, json, os, sys
r = subprocess.run('python bac/evaluate.py --name smoke --limit 5 --out /kaggle/tmp/smoke', shell=True, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-3000:])
assert os.path.exists('/kaggle/tmp/smoke/smoke_val.json'), 'SMOKE TEST FAILED: evaluation does not run - see error above'
print('smoke eval OK, gIoU on 5 images:', round(json.load(open('/kaggle/tmp/smoke/smoke_val.json'))['gIoU'], 3))

In [ ]:
r = subprocess.run('python bac/train.py --name smoke --kind bac --tokens 64 --steps 3', shell=True, capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-3000:])
assert os.path.exists('runs/smoke/connector.pt'), 'SMOKE TEST FAILED: training does not run - see error above'
import shutil; shutil.rmtree('runs/smoke'); print('smoke train OK')

In [ ]:
!python bac/test_connectors.py 2>&1 | grep -E "OK|passed|Error"

In [ ]:
# fine-tuning smoke test: 3 steps, then evaluation with the weights (plain and 8-way TTA) on a few images
for cmd in ['PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python bac/train.py --name smoke_ft --kind released --mask-path --aug d4 --steps 3 --batch 1 --accum 2',
            'python bac/evaluate.py --name s1 --weights runs/smoke_ft/connector.pt --limit 3 --out /kaggle/tmp/sft',
            'python bac/evaluate.py --name s8 --weights runs/smoke_ft/connector.pt --limit 2 --tta 8 --subset calib300 --out /kaggle/tmp/sft']:
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(cmd, '->', r.returncode); print(r.stdout[-1200:], r.stderr[-2500:])
assert os.path.exists('/kaggle/tmp/sft/s1_val.json') and os.path.exists('/kaggle/tmp/sft/s8_val.json'), 'FT SMOKE FAILED'
shutil.rmtree('runs/smoke_ft'); print('ft smoke OK')

In [ ]:
!bash kaggle/ft_queue.sh

In [ ]:
import json, glob
for p in sorted(glob.glob('results/*_val.json')):
    d = json.load(open(p)); print(d['name'], d['n'], round(100 * d['gIoU'], 2), round(100 * d['cIoU'], 2))
rows = glob.glob('results/*_val.json')

In [ ]:
!cd /kaggle/working && zip -qr bac_ft_results.zip results && ls -la bac_ft_results.zip
!mkdir -p /kaggle/working/runs && cp -r /kaggle/tmp/Minor/runs/ft_* /kaggle/working/runs/ 2>/dev/null; du -sh /kaggle/working/runs
assert rows, 'NO RESULTS'